# BONUS: Troubleshooting, Monitoring & Optimization

This bonus module covers the skills tested in **DEA Exam Domain 6: Troubleshooting, Monitoring, and Optimization** — Spark UI analysis, job failure diagnosis, and cluster startup issues.

> This is a BONUS module — cover it if time permits, or use it as self-study material.

| Section | Topic | Exam Keywords |
|---------|-------|---------------|
| 1 | Spark UI Overview | Jobs, Stages, Storage, Executors, SQL tab |
| 2 | Job Failure Diagnosis | AnalysisException, OOM, FileNotFound, VACUUM |
| 3 | Cluster Startup Failures | init script, driver OOM, library conflict |
| 4 | Lakeflow Job Monitoring | Repair Run, run history, task error |


## Section 1: Spark UI Overview

For a full walkthrough of Spark UI tabs and key Stages metrics, see **[01 — Platform & Workspace → Section 5](../../day1/demo/01_platform_and_workspace.ipynb)**.

The code cell below lets you inspect the current Spark configuration, including AQE flags and memory settings — useful for live diagnostics.


In [ ]:
# Inspect current Spark configuration — useful for verifying AQE, broadcast threshold, etc.
print('Spark version:', spark.version)
print()

keys = [
    'spark.sql.adaptive.enabled',
    'spark.sql.adaptive.coalescePartitions.enabled',
    'spark.sql.adaptive.skewJoin.enabled',
    'spark.sql.autoBroadcastJoinThreshold',
    'spark.sql.shuffle.partitions',
    'spark.executor.memory',
    'spark.driver.memory',
]
for k in keys:
    v = spark.conf.get(k, '(not set)')
    print(f'{k:55s} = {v}')

## Section 2: Job Failure Diagnosis

When a Databricks job fails, the run UI shows the **error type and stack trace** for each failed task. Knowing how to read these errors quickly is a core exam skill.

### Where to find error details
1. **Databricks Workflows** -> select the job run -> click the failed task (red)
2. In the task detail panel, expand the **Error** section
3. The first line usually contains the exception class — the most important part

### Common failure types

| Error | Root cause | Fix |
|-------|-----------|-----|
| `AnalysisException: Table or view not found` | Wrong catalog/schema context | Add `USE CATALOG` / `USE SCHEMA`, or use three-part name |
| `AnalysisException: Column not found` | Schema mismatch or typo | Check column name; verify schema evolution settings |
| `java.lang.OutOfMemoryError` | Driver or executor ran out of heap | Increase memory, reduce broadcast threshold, increase shuffle partitions |
| `ClassNotFoundException` / `NoClassDefFoundError` | Library version conflict | Pin library versions; use cluster-scoped init scripts |
| `FileNotFoundException` / `Path does not exist` | Volume unmounted, wrong path, or file deleted | Verify path; check Volume mount; re-run upstream pipeline |
| `DELTA_CONCURRENT_DELETE_READ` | VACUUM deleted files still in a Time Travel query | Increase `delta.deletedFileRetentionDuration`; avoid aggressive VACUUM intervals |
| `SparkException: Task failed while writing rows` | Executor crash during write | Check executor OOM or disk space; increase node size |
| `StreamingQueryException` | Streaming query aborted | Check checkpoint directory; verify source schema has not changed |

In [ ]:
# ── Reproduce and diagnose AnalysisException: Table not found ─────────────────
try:
    spark.sql('SELECT * FROM nonexistent_catalog.nonexistent_schema.nonexistent_table LIMIT 1')
except Exception as e:
    error_type = type(e).__name__
    first_line = str(e).splitlines()[0]
    print(f'Error type : {error_type}')
    print(f'First line : {first_line}')
    print()
    print('Fix: use three-part name or set context:')
    print('  USE CATALOG my_catalog;')
    print('  USE SCHEMA  my_schema;')
    print('  SELECT * FROM my_table;')

In [ ]:
# ── OutOfMemoryError: configuration levers ────────────────────────────────────
print('Current autoBroadcastJoinThreshold:',
      spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))

# Disable auto-broadcast to prevent accidental OOM from large broadcast
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
print('After disabling broadcast:',
      spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))

# Re-enable with a safe threshold (10 MB)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
print('After setting 10 MB threshold:',
      spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))

print()
print('OOM fix checklist:')
print('  1. Check if a broadcast join is the cause (Spark UI > SQL > physical plan)')
print('  2. Lower autoBroadcastJoinThreshold to prevent accidental large broadcasts')
print('  3. Increase executor memory in cluster config (worker node type)')
print('  4. Increase spark.sql.shuffle.partitions to reduce per-partition size')

In [ ]:
# ── DELTA_CONCURRENT_DELETE_READ: Time Travel window exceeded ─────────────────
msg = (
    'Error: DELTA_CONCURRENT_DELETE_READ\n'
    'Cause: VACUUM deleted files that are still referenced by an older table version.\n'
    '\n'
    'Timeline:\n'
    '  t=0   Table written at version 10\n'
    '  t=1h  VACUUM runs with retentionHours=0 (violates 7-day default)\n'
    '  t=1h  User queries: SELECT * FROM table VERSION AS OF 8\n'
    '  -> Spark tries to read files deleted by VACUUM -> DELTA_CONCURRENT_DELETE_READ\n'
    '\n'
    'Fix:\n'
    '  1. Never set retentionHours below 168 (7 days) unless no consumers use Time Travel\n'
    "  2. Set a safe retention:\n"
    "       ALTER TABLE my_table\n"
    "       SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 14 days');\n"
    '  3. Run VACUUM without DRY RUN only after verifying no active Time Travel queries\n'
)
print(msg)

## Section 3: Cluster Startup Failures

Cluster startup failures occur before any user code runs. They are diagnosed through the **Cluster Event Log**, not the notebook output.

### How to access the Cluster Event Log
1. Go to **Compute** -> select the cluster
2. Click the **Event log** tab
3. Filter by event type: **Error** or **Init script**

### Init script failures
- An init script is a shell script that runs on every node at cluster start
- If it exits with a non-zero code, the cluster fails to start
- **Fix:** Check the init script log (linked in the Event Log) for the exact shell error

### Library installation failures
- Libraries installed via the cluster Libraries tab run `pip install` during startup
- Conflicts between library versions cause `ERROR: pip's dependency resolver` messages
- **Fix:** Pin exact library versions; use `%pip install` in the notebook instead (notebook-scoped libraries avoid conflicts)

### Driver OOM at startup
- The driver starts but crashes immediately with `OutOfMemoryError` before running user code
- **Cause:** A large init script loads data into driver memory; or the driver node type is too small
- **Fix:** Increase driver node type; remove init tasks that load large data into memory

### Exam tip: Init scripts vs cluster libraries

| Feature | Init Script | Cluster Library |
|---------|------------|------------------|
| Runs as | Shell script | `pip install` / Maven |
| Installed on | All nodes | All nodes |
| Notebook-scoped? | No | No (use `%pip`) |
| Failure visibility | Event Log -> init script log | Event Log -> pip output |

In [ ]:
# ── Notebook-scoped library install (avoids cluster-level conflicts) ───────────
# Use %pip in a notebook cell to install libraries scoped to the current session.
# This does NOT require a cluster restart and does NOT affect other notebooks.

# Example (commented out):
# %pip install faker==20.1.0

# After %pip install, the kernel restarts automatically and variables are reset.
# Use dbutils.library.restartPython() if you need to control the restart point.

print('Best practice: prefer %pip in notebook cells over cluster-level library installs.')
print('This avoids version conflicts between different notebooks on the same cluster.')

## Section 4: Lakeflow Job Monitoring

Databricks Lakeflow (Workflows) provides a rich UI for monitoring and recovering from job failures.

### Repair Run
When a multi-task job partially fails, you do not need to re-run the entire job.

- Click **Repair Run** on the failed run
- Databricks re-runs only:
  - The **failed tasks**
  - All **downstream tasks** that depend on the failed tasks
- Successfully completed tasks are **not re-run** (their outputs are reused)

This is the recommended recovery mechanism — it saves time and avoids re-processing data already written.

### Run History Filters
The job **Run History** page supports filtering by:
- **Status**: Succeeded, Failed, Running, Canceled
- **Date range**: narrow to a specific time window
- **Task**: filter runs where a specific task failed

### Reading Task Error Details
1. Click the failed run in the Run History
2. In the task graph, click the **red task node**
3. Expand the **Error** section in the right panel
4. The first line is the exception class; the stack trace shows the exact cell and line

### Job Run States

| State | Meaning |
|-------|--------|
| **Running** | Currently executing |
| **Succeeded** | All tasks completed successfully |
| **Failed** | One or more tasks failed |
| **Canceled** | Manually stopped or timed out |
| **Skipped** | Task was skipped due to conditional dependency |
| **Blocked** | Upstream task has not completed yet |

In [ ]:
# ── List recent job runs using the Databricks SDK ────────────────────────────
try:
    from databricks.sdk import WorkspaceClient
    import datetime
    w = WorkspaceClient()
    runs = list(w.jobs.list_runs(limit=10))
    if runs:
        print(f"{'Run ID':<12} {'Job ID':<12} {'Status':<15} {'Start Time'}")
        print('-' * 65)
        for r in runs:
            start = datetime.datetime.fromtimestamp(r.start_time / 1000).strftime('%Y-%m-%d %H:%M')
            state = r.state.life_cycle_state.value if r.state else 'UNKNOWN'
            print(f'{r.run_id:<12} {r.job_id:<12} {state:<15} {start}')
    else:
        print('No recent runs found.')
except Exception as e:
    print(f'SDK not available or not authenticated: {e}')
    print('In a live Databricks environment, this cell lists recent job runs.')

In [ ]:
# ── Troubleshooting decision guide ───────────────────────────────────────────
guide = (
    'SYMPTOM                              FIRST PLACE TO LOOK           LIKELY FIX\n'
    '=' * 90 + '\n'
    'Job task failed                      Run details > Error section   Fix query; check schema/path\n'
    'One task much slower than others     Spark UI > Stages > Tasks     Data skew: AQE or salting\n'
    'High shuffle read/write              Spark UI > Stages             Broadcast join; fewer groupBy\n'
    'Spill metrics non-zero               Spark UI > Stages > Metrics   More partitions; more memory\n'
    'Cluster not starting                 Compute > Event Log           Init script error; OOM\n'
    'Library version error                Compute > Event Log > pip     Pin versions; use %pip\n'
    'AnalysisException: not found         Error first line              USE CATALOG/SCHEMA; 3-part name\n'
    'OutOfMemoryError                     Executor/Driver logs          Lower broadcast; more memory\n'
    'DELTA_CONCURRENT_DELETE_READ         Delta log / VACUUM history    Increase retention; no vacuum 0h\n'
    'Streaming query stopped              StreamingQueryException       Reset checkpoint; fix schema\n'
)
print(guide)

← [11 — Exam Preparation](11_exam_preparation.ipynb) | **[README](../../../README.md)**